# Faster R-CNN Zindi submission

This notebook loads the selected Faster R-CNN checkpoint, predicts one turtle-face box for each ID in `SampleSubmission.csv`, validates the competition schema, and writes a local CSV. It does not train or upload a model.

The detector output is reduced to the highest-confidence class-1 detection after NMS. The box is mapped back through the letterbox transform and written as normalized `x, y, w, h` in the original image frame. A missing valid detection is represented by four zeros and counted in coverage.

In [ ]:
from pathlib import Path
import sys

# Locate the repository whether Jupyter starts in the project or its parent folder.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'data' / 'raw' / 'SampleSubmission.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project root or its parent directory.')

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Project root: {PROJECT_ROOT.resolve()}')

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageDraw

from turtle_detection.data_pipeline import DEFAULT_TARGET_SIZE
from turtle_detection.detection import build_faster_rcnn_resnet50_fpn
from turtle_detection.submission import (
    load_submission_inputs,
    predict_faster_rcnn_submission,
    write_submission_csv,
)

SEED = 42
TARGET_SIZE = DEFAULT_TARGET_SIZE
INFERENCE_BATCH_SIZE = 2
SAMPLE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'SampleSubmission.csv'
IMAGE_DIR = PROJECT_ROOT / 'data' / 'raw' / 'IMAGES_512'
CHECKPOINT_PATH = (
    PROJECT_ROOT
    / 'outputs'
    / 'checkpoints'
    / 'fasterrcnn_resnet50_fpn_seed42_6f7eda828cdf4753a12a8482c195a529_best.pt'
)
OUTPUT_PATH = PROJECT_ROOT / 'outputs' / 'submissions' / 'fasterrcnn_resnet50_fpn_seed42.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Model checkpoint not found: {CHECKPOINT_PATH}')
print({
    'device': str(DEVICE),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'target_size': TARGET_SIZE,
    'batch_size': INFERENCE_BATCH_SIZE,
    'checkpoint': CHECKPOINT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    'output': OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(),
})

In [ ]:
# Rebuild the same detector configuration used by the trained checkpoint, without downloading weights.
model = build_faster_rcnn_resnet50_fpn(
    pretrained=False,
    trainable_backbone_layers=3,
    min_size=TARGET_SIZE[0],
    max_size=TARGET_SIZE[1],
    box_score_thresh=0.0,
    box_nms_thresh=0.5,
    box_detections_per_img=100,
)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if 'state_dict' not in checkpoint:
    raise ValueError('Checkpoint does not contain a detector state_dict.')
model.load_state_dict(checkpoint['state_dict'], strict=True)
model.to(DEVICE).eval()
print({
    'checkpoint_epoch': checkpoint.get('epoch'),
    'validation_metrics': checkpoint.get('metrics'),
    'parameter_tensors': len(checkpoint['state_dict']),
})

# Preserve the sample row order and fail before inference if any ID is missing or ambiguous.
sample, image_paths = load_submission_inputs(SAMPLE_PATH, IMAGE_DIR)
print(f'Validated {len(sample)} unique IDs and resolved one image per ID.')

In [ ]:
# Run inference only; each model output is mapped back to original-image normalized coordinates.
started_at = time.perf_counter()
predictions, diagnostics = predict_faster_rcnn_submission(
    model=model,
    sample=sample,
    image_paths=image_paths,
    device=DEVICE,
    target_size=TARGET_SIZE,
    batch_size=INFERENCE_BATCH_SIZE,
)
elapsed_seconds = time.perf_counter() - started_at

# Validate against the original IDs before writing; the CSV writer also omits the dataframe index.
sample_ids = sample['Image_ID'].astype(str).tolist()
submission_path = write_submission_csv(predictions, OUTPUT_PATH, expected_ids=sample_ids)
no_detection_count = int((~diagnostics['has_detection']).sum())
coverage = float(diagnostics['has_detection'].mean())
print({
    'submission_path': str(submission_path),
    'rows': len(predictions),
    'prediction_coverage': coverage,
    'no_detection_count': no_detection_count,
    'elapsed_seconds': round(elapsed_seconds, 1),
})
display(predictions.head(10))

In [ ]:
# Inspect a deterministic mixture of detected and missed cases before using the file.
preview = diagnostics.sort_values(['has_detection', 'confidence'], ascending=[True, True]).head(4)
detected_preview = diagnostics[diagnostics['has_detection']]
if not detected_preview.empty:
    preview = pd.concat([preview, detected_preview.sample(min(4, len(detected_preview)), random_state=SEED)])
preview = preview.drop_duplicates('Image_ID').head(8)
display(preview[['Image_ID', 'has_detection', 'confidence', 'class_one_detections', 'x', 'y', 'w', 'h']])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis, (_, row) in zip(axes.flat, preview.iterrows()):
    image_index = sample_ids.index(str(row['Image_ID']))
    with Image.open(image_paths[image_index]) as opened_image:
        image = opened_image.convert('RGB')
    draw = ImageDraw.Draw(image)
    width, height = image.size
    if row['has_detection']:
        box = (row['x'] * width, row['y'] * height, (row['x'] + row['w']) * width, (row['y'] + row['h']) * height)
        draw.rectangle(box, outline='red', width=3)
        label = f"score={row['confidence']:.3f}"
    else:
        label = 'no valid detection'
    axis.imshow(image)
    axis.set_title(f"{row['Image_ID']} | {label}")
    axis.axis('off')
for axis in axes.flat[len(preview):]:
    axis.axis('off')
plt.tight_layout()